In [ ]:
import numpy as np
import random

# 第20章 强化学习的挑战与未来

## 目录
1. 样本效率
2. 计算资源
3. 挑战与未来方向
4. 编程题

---
## 1. 样本效率

In [ ]:
"""样本效率"""
def sample_complexity(gamma, epsilon):
    """样本复杂度O(1/(1-gamma)^2/epsilon^2)"""
    return 1 / ((1 - gamma) ** 2 * epsilon ** 2)

print("样本效率: 样本复杂度分析")

---
## 2. 计算资源

In [ ]:
import time
def compute_benchmark():
    """计算基准测试"""
    n_ops = 1000000
    start = time.time()
    _ = np.random.randn(100, 100) @ np.random.randn(100, 100)
    elapsed = time.time() - start
    print(f"矩阵乘法(100x100): {elapsed*1000:.2f}ms")

compute_benchmark()

---
## 3. 未来方向

In [ ]:
def print_future_directions():
    """打印未来研究方向"""
    directions = [
        "离线强化学习 (Offline RL)",
        "元学习 (Meta-RL)",
        "多智能体强化学习 (MARL)",
        "安全强化学习 (Safe RL)",
        "可解释强化学习"
    ]
    for d in directions:
        print(f"  - {d}")

print_future_directions()

---
## 4. 编程题

### 编程题1：实现离线强化学习的简化版本(CQL)

In [ ]:
class CQL:
    """Conservative Q-Learning (简化)"""
    def __init__(self, n_states, n_actions, alpha=0.001):
        self.q_table = np.zeros((n_states, n_actions))
        self.alpha = alpha
        self.gamma = 0.99
    
    def get_q(self, state, action):
        return self.q_table[state, action]
    
    def update(self, batch, dataset_size, beta=0.5):
        """保守更新"""
        for s, a, r, ns, done in batch:
            target = r + self.gamma * (0 if done else np.max(self.q_table[ns]))
            old_q = self.q_table[s, a]
            self.q_table[s, a] += self.alpha * (target - old_q)
        
        "保守惩罚"""
        expected_q = np.mean(self.q_table)
        max_q = np.max(self.q_table)
        cql_loss = beta * (max_q - expected_q)
        
        return cql_loss

class OfflineDataset:
    """离线数据集"""
    def __init__(self):
        self.data = []
        
    def add(self, transition):
        self.data.append(transition)
    
    def sample(self, batch_size):
        return random.sample(self.data, batch_size)

class SimpleEnv:
    """简单环境"""
    def __init__(self, size=4):
        self.size = size
        self.pos = 0
    def reset(self): self.pos = 0; return self.pos
    def step(self, a):
        if a == 1: self.pos = min(self.size-1, self.pos + 1)
        reward = 1.0 if self.pos == self.size-1 else -0.01
        return self.pos, reward, self.pos == self.size-1

def train_offline():
    """训练离线强化学习"""
    cql = CQL(n_states=4, n_actions=2)
    dataset = OfflineDataset()
    env = SimpleEnv()
    
    # 收集数据(可能有策略外数据)
    for _ in range(500):
        s = env.reset()
        for _ in range(10):
            a = random.randint(0, 1)
            ns, r, done = env.step(a)
            dataset.add((s, a, r, ns, done))
            s = ns
            if done: break
    
    # 离线训练
    for ep in range(50):
        batch = dataset.sample(32)
        loss = cql.update(batch, len(dataset.data))
    
    print("CQL离线训练:")
    print(f"  Q值表: {cql.q_table}")

train_offline()

---

### 编程题2：实现元强化学习的简化版本(MAML)

In [ ]:
class MAML:
    """MAML简化版"""
    def __init__(self, n_states, n_actions, inner_lr=0.1, outer_lr=0.01):
        self.policy = np.zeros((n_states, n_actions))
        self.inner_lr = inner_lr
        self.outer_lr = outer_lr
    
    def select_action(self, state):
        return np.argmax(self.policy[state])
    
    def inner_update(self, batch):
        """内部更新(快速适应)"""
        temp_policy = self.policy.copy()
        for s, a, r in batch:
            temp_policy[s, a] += self.inner_lr * r
        return temp_policy
    
    def outer_update(self, adapted_policy, original_rewards):
    """外部更新(元学习)"""
        for s, a, r in original_rewards:
            self.policy[s, a] += self.outer_lr * r

def meta_train():
    """元训练"""
    maml = MAML(n_states=4, n_actions=2)
    
    tasks = [
        [(0, 1, 1.0), (1, 1, 1.0), (2, 1, 1.0)],
        [(0, 0, 1.0), (1, 0, 1.0), (2, 0, 1.0)],
    ]
    
    for ep in range(50):
        for task in tasks:
            adapted = maml.inner_update(task[:2])
            maml.outer_update(task[2:])
    
    print("MAML元学习:")
    print(f"  元策略: {maml.policy[0]}")

meta_train()

---

### 编程题3：实现安全强化学习的约束MPO

In [ ]:
class ConstrainedMPO:
    """约束MPO (简化版)"""
    def __init__(self, n_states, n_actions, constraint_threshold=0.1):
        self.policy = np.zeros((n_states, n_actions))
        self.cost_table = np.zeros((n_states, n_actions))
        self.constraint = constraint_threshold
    
    def select_action(self, state):
        probs = np.exp(self.policy[state] - np.max(self.policy[state]))
        return np.random.choice(len(probs), p=probs/np.sum(probs))
    
    def update(self, s, a, reward, cost):
        """约束更新"""
        self.policy[s, a] += 0.01 * reward
        self.cost_table[s, a] += 0.01 * cost
        
        if cost > self.constraint:
            self.policy[s, a] -= 0.005 * cost

def compare_safe():
    """对比安全RL"""
    safe_agent = ConstrainedMPO(n_states=4, n_actions=2)
    normal_agent = MAML(n_states=4, n_actions=2)
    
    costs = [0.05, 0.15, 0.25]
    safe_costs, normal_costs = [], []
    
    for cost_threshold, agent_costs in [(0.1, safe_costs), (0.5, normal_costs)]:
        agent = safe_agent if cost_threshold < 0.2 else ConstrainedMPO(4, 2, 0.5)
        
        for cost in costs:
            if cost <= cost_threshold: agent_costs.append(1)
            else: agent_costs.append(-1)
    
    print("安全强化学习对比:")
    print(f"  约束(<0.1): {sum(safe_costs)/len(safe_costs):.1%}")
    print(f"  宽松(<0.5): {sum(normal_costs)/len(normal_costs):.1%}")

compare_safe()

print("\n第20章 强化学习的挑战与未来 - 编程题完成!")